# Lecture 5.7 — Parallelisation with asyncio.gather

**Section 05 — Multi-Agent Orchestration & Guardrails**

In this notebook you will run independent agent calls at the same time with
`asyncio.gather` instead of one after another, and see the two big reasons to
do that: saving wall-clock time, and generating several candidate answers so
you can pick the best one.

## Cell 1: Install the OpenAI Agents SDK

This notebook uses the **OpenAI Agents SDK**, the Python framework this
course is built on. The cell below installs it into the current Colab
runtime.

The version is pinned so that the code in this notebook behaves the same way
it does in the recording. If the package is already installed in this
session, pip notices that and the cell finishes quickly without reinstalling
anything.

Run this cell first, and let it finish before moving on.

In [1]:
# Pinned for reproducibility. To use the latest version,
# run: pip install openai-agents
# Or substitute your preferred version below.
!pip install openai-agents==0.18.3 -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 880.8/880.8 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.6/142.6 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 222.6/222.6 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 4.3 MB/s eta 0:00:00


## Cell 2: Configure your OpenAI API key

The SDK reads your OpenAI API key from the `OPENAI_API_KEY` environment
variable. In Colab, the safest way to provide it is **Colab Secrets**, which
keeps the key out of the notebook itself.

**Steps to add your key as a Colab secret:**

1. Click the key icon (🔑) in the left sidebar of Colab.
2. Click **Add new secret**.
3. Set the name to `OPENAI_API_KEY` exactly.
4. Paste your API key as the value.
5. Toggle **Notebook access** on for this notebook.

The cell below reads that secret with `userdata.get()` and writes it into
`os.environ`, which is where the SDK looks for it.

**Running locally instead of Colab?** Set the environment variable in your
terminal before starting Python, for example `export OPENAI_API_KEY=sk-...`
on macOS or Linux, and skip the `userdata` import.

In [2]:
import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

## Cell 3: Set the model name

Every agent in this notebook is built with the same `MODEL_NAME` variable
instead of a hardcoded model string. Changing this one line changes the
model used everywhere in the notebook.

In [3]:
# See latest models at: https://platform.openai.com/docs/models
MODEL_NAME = "gpt-5.4-mini"

## Cell 4: Imports

| Import | Purpose in this notebook |
|---|---|
| `asyncio` | Standard library. Provides `asyncio.gather`, the function this whole lecture is about. |
| `time` | Standard library. Used to time the sequential and parallel runs so you can compare them directly. |
| `dataclass` | Used later in the notebook to build a plain context object for the concurrency-safety demo. |
| `Reasoning` | Configures `reasoning.effort` on an agent's model settings. |
| `Agent` | Defines each translator agent and the picker agent. |
| `ItemHelpers` | Extracts plain text from an agent's run items, already familiar from Lecture 4.1. |
| `ModelSettings` | Sets `reasoning` and `verbosity` on each agent. |
| `RunConfig` | Carries a shared `group_id` so parallel runs trace as one workflow. |
| `Runner` | Executes each agent with `await Runner.run(...)`. |

`asyncio` and `time` are standard library modules, not part of the Agents
SDK. They do the actual work of running things concurrently and measuring
how long that takes.

In [4]:
import asyncio
import time
from dataclasses import dataclass

from openai.types.shared import Reasoning

from agents import (
    Agent,
    ItemHelpers,
    ModelSettings,
    RunConfig,
    Runner,
)

## Cell 5: Why parallelise

Lecture 5.6 covered code-driven patterns where each agent's output feeds the
next one; the steps genuinely depend on each other, so they have to run in
order. This lecture covers the fourth and final code-driven pattern: agent
calls that **do not** depend on each other at all.

When two or more `Runner.run()` calls have nothing to do with one another,
running them one after another wastes real time. Python's own
`asyncio.gather` primitive lets you fire them all off at once and wait for
all of them to finish together.

There are two distinct reasons to reach for this pattern:

| Motivation | What it looks like |
|---|---|
| **Speed** | Independent tasks run concurrently instead of queueing behind each other. Three two-second calls take about two seconds total instead of six. |
| **Quality through redundancy** | Generate several candidate responses to the same prompt, then have a separate agent pick the best one. |

Both motivations show up in this notebook: first as a straightforward speed
comparison, then as a "generate three, pick the best" pattern, and finally
as a fan-out across three genuinely different specialist agents.

## Cell 6: Sequential baseline — measuring the cost of not parallelising

Before introducing `asyncio.gather`, it helps to see the problem it solves.
`spanish_agent` below is a plain translator with no tools and no handoffs.
The cell runs it three times, back to back, on the exact same message, and
times the whole thing with `time.time()`.

| Setting | Value | Why |
|---|---|---|
| `model` | `MODEL_NAME` | Keeps the model consistent with the rest of the notebook. |
| `reasoning=Reasoning(effort="none")` | Translation doesn't need extended reasoning, so effort is turned off for lower latency. |
| `verbosity="low"` | Keeps the translated output short and to the point. |

Each `await Runner.run(...)` call here waits for its response before the
next one starts. Three genuinely independent translation requests are being
forced through one at a time, purely because of how the code is written, not
because the task requires it.

Run this cell and read the printed elapsed time. You'll compare it against
the parallel version in the next cell.

In [5]:
spanish_agent = Agent(
    name="Spanish Agent",
    instructions="You translate the user's message to Spanish.",
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
    ),
)

msg = "Good morning! How is your day going?"

start = time.time()
res_1 = await Runner.run(spanish_agent, msg)
res_2 = await Runner.run(spanish_agent, msg)
res_3 = await Runner.run(spanish_agent, msg)
elapsed = time.time() - start

print(f"Sequential: {elapsed:.1f}s for 3 runs")

Sequential: 7.7s for 3 runs


## Cell 7: Parallel execution with asyncio.gather

`asyncio.gather(*coroutines)` starts every coroutine passed to it at the same
time, waits for all of them to finish, and returns their results in a list
in the **same order the coroutines were passed in**, regardless of which one
actually finished first.

Two mechanics matter here:

| Mechanic | What it means |
|---|---|
| **Result order** | `res_1, res_2, res_3` line up with the order the calls were written, not completion order. |
| **Elapsed time** | Wall-clock time tracks the **slowest** individual call, not the sum of all three. |

Each `Runner.run()` call also receives a `RunConfig` with the same
`group_id`. That doesn't change what the agents do; it links all three
parallel runs together under one workflow in the traces dashboard, the same
`group_id` pattern from Lecture 4.4 and Lecture 5.6.

Once all three results are back, `ItemHelpers.text_message_outputs()`, which
you've already used in Lecture 4.1 and Lecture 5.6, pulls the plain text out
of each result's `new_items` list.

Run this cell and compare its printed time against Cell 6's. Watch how much
closer it is to the time of a single call than to the sum of three.

In [6]:
start = time.time()
res_1, res_2, res_3 = await asyncio.gather(
    Runner.run(
        spanish_agent,
        msg,
        run_config=RunConfig(
            workflow_name="Parallel translation",
            group_id="translate-run-001",
        ),
    ),
    Runner.run(
        spanish_agent,
        msg,
        run_config=RunConfig(
            workflow_name="Parallel translation",
            group_id="translate-run-001",
        ),
    ),
    Runner.run(
        spanish_agent,
        msg,
        run_config=RunConfig(
            workflow_name="Parallel translation",
            group_id="translate-run-001",
        ),
    ),
)
elapsed = time.time() - start

print(f"Parallel: {elapsed:.1f}s for 3 runs")

outputs = [
    ItemHelpers.text_message_outputs(res_1.new_items),
    ItemHelpers.text_message_outputs(res_2.new_items),
    ItemHelpers.text_message_outputs(res_3.new_items),
]

for i, out in enumerate(outputs, 1):
    print(f"Translation {i}: {out}")

Parallel: 1.1s for 3 runs
Translation 1: ¡Buenos días! ¿Cómo va tu día?
Translation 2: ¡Buenos días! ¿Cómo va tu día?
Translation 3: ¡Buenos días! ¿Cómo va tu día?


## Cell 8: Selecting the best result — the redundancy pattern

`outputs` from Cell 7 now holds three independent Spanish translations of the
same sentence. `translation_picker` is a second agent whose only job is to
read a set of candidate translations and choose the best one.

This is the "quality through redundancy" motivation from Cell 5 in action:
instead of trusting a single generation, you generate several and let a
dedicated agent judge between them. It costs three extra model calls, but it
is one of the simplest ways to raise output quality without touching a
single prompt.

The three candidates are joined into one string and handed to
`translation_picker` along with the original message, so it has everything
it needs to judge fairly. The same `group_id` is reused here too, so all
four runs, the three translators and the picker, trace as one connected
workflow.

Run this cell and read `best_translation.final_output`. That's the picker's
verdict, not a translation you generated yourself.

In [7]:
translation_picker = Agent(
    name="Translation Picker",
    instructions=(
        "You pick the best Spanish translation from the given options."
    ),
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
    ),
)

translations = "\n\n".join(outputs)

best_translation = await Runner.run(
    translation_picker,
    f"Input: {msg}\n\nTranslations:\n{translations}",
    run_config=RunConfig(
        workflow_name="Parallel translation",
        group_id="translate-run-001",
    ),
)

print("Best translation:", best_translation.final_output)

Best translation: ¡Buenos días! ¿Cómo va tu día?


## Cell 9: Parallel fan-out with different agents — not just repeats

Every parallel call so far has used the same `spanish_agent`. That's not a
requirement of `asyncio.gather`. It just fires whatever coroutines you give
it, and there's nothing stopping those coroutines from wrapping completely
different agents.

This cell defines three specialists, `french_agent`, `german_agent`, and
`japanese_agent`, each translating the same English sentence into a
different language. None of their outputs depend on each other, so they're
just as safe to parallelise as three copies of the same agent were.

| Agent | Task |
|---|---|
| `french_agent` | Translate to French |
| `german_agent` | Translate to German |
| `japanese_agent` | Translate to Japanese |

Run this cell and compare the elapsed time to what three sequential calls to
three different agents would have cost. Three unrelated jobs, done by three
different specialists, still finish in roughly the time of the slowest one.

In [8]:
english_text = "Please confirm your appointment for tomorrow at 3pm."

french_agent = Agent(
    name="French Agent",
    instructions="You translate the user's message to French.",
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
    ),
)

german_agent = Agent(
    name="German Agent",
    instructions="You translate the user's message to German.",
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
    ),
)

japanese_agent = Agent(
    name="Japanese Agent",
    instructions="You translate the user's message to Japanese.",
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
    ),
)

start = time.time()
fr_result, de_result, ja_result = await asyncio.gather(
    Runner.run(french_agent, english_text),
    Runner.run(german_agent, english_text),
    Runner.run(japanese_agent, english_text),
)
elapsed = time.time() - start

print(f"Three-language fan-out: {elapsed:.1f}s")
print("French:", fr_result.final_output)
print("German:", de_result.final_output)
print("Japanese:", ja_result.final_output)

Three-language fan-out: 1.1s
French: Veuillez confirmer votre rendez-vous pour demain à 15 h.
German: Bitte bestätigen Sie Ihren Termin für morgen um 15 Uhr.
Japanese: 明日の午後3時のご予約をご確認ください。


## Cell 10: Context safety in concurrent runs — critical warning

Lecture 4.5 covered `RunContextWrapper` and the context object it wraps as a
plain local Python object your tools and hooks can read and write. That
object is **not thread-safe**. `asyncio.gather` runs several `Runner.run()`
calls concurrently, and if they all mutate one shared context instance at
the same time, you're inviting a race condition on that shared state.

`SharedCounter` below is a minimal example context. The commented-out block
shows the pattern to avoid: one `shared_ctx` instance passed into every
concurrent run. The working code beneath it creates a **separate** context
instance, `ctx_1`, `ctx_2`, `ctx_3`, for each concurrent run instead.

| Pattern | Safe? |
|---|---|
| One shared mutable context passed to every concurrent `Runner.run()` | No. Risks race conditions on shared state. |
| A fresh context instance per concurrent `Runner.run()` | Yes. Each run's state is fully isolated. |

Run this cell. There's nothing to compare output-wise here; the point is the
pattern itself, not what gets printed.

In [9]:
@dataclass
class SharedCounter:
    count: int = 0

# WRONG PATTERN — DO NOT DO THIS (shown as comment only)
# shared_ctx = SharedCounter()
# await asyncio.gather(
#     Runner.run(spanish_agent, msg, context=shared_ctx),
#     Runner.run(spanish_agent, msg, context=shared_ctx),
# )
# Sharing one mutable context instance across concurrent
# runs risks race conditions.

# CORRECT PATTERN — separate context instance per run
ctx_1 = SharedCounter()
ctx_2 = SharedCounter()
ctx_3 = SharedCounter()

results = await asyncio.gather(
    Runner.run(spanish_agent, msg, context=ctx_1),
    Runner.run(spanish_agent, msg, context=ctx_2),
    Runner.run(spanish_agent, msg, context=ctx_3),
)

print("Each run had its own isolated context instance.")

Each run had its own isolated context instance.


## Cell 11: Timing comparison summary

The table below generalises what Cells 6 and 7 demonstrated, using three
independent two-second calls as a round-number example.

| Approach | 3 independent 2-second calls | Formula |
|---|---|---|
| Sequential (`await` one at a time) | ~6 seconds | Sum of all call times |
| Parallel (`asyncio.gather`) | ~2 seconds | Max of all call times |

The gap only widens as you add more independent calls. Ten sequential
two-second calls take twenty seconds; ten parallel ones still take about two.

## Cell 12: When to reach for asyncio.gather

`asyncio.gather` is not a replacement for every multi-agent pattern in this
section. Use the table below to decide whether it fits the task in front of
you.

| Use `asyncio.gather` when | Don't use it when |
|---|---|
| Tasks are genuinely independent; no task needs another's output | One task's output feeds another (use 5.6's sequential pattern instead) |
| You want several candidate responses to pick from | You need a single deterministic answer, not options |
| Latency matters and tasks can run concurrently | The API or model has strict rate limits that parallel calls would exceed |
| Each task gets its own context instance | Tasks need to share and mutate the same context safely |

That decision, independent versus dependent tasks, is the same question
you'll keep asking throughout Section 05 as you meet guardrails next,
starting with input guardrails in Lecture 5.8.